In [1]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Drop detection utilities
# ------------------------------------------------------------

def detect_missing_indices(df: pd.DataFrame, fps: float, tolerance_s: float, ts_col="Timestamp") -> list[int]:
    expected_dt = 1.0 / fps
    df = df.copy()
    df[ts_col] = pd.to_datetime(df[ts_col], utc=True)
    df = df.sort_values(ts_col).reset_index(drop=True)

    dt = df[ts_col].diff().dt.total_seconds()
    n_dropped_est = np.round(dt / expected_dt - 1).clip(lower=0).astype("Int64")

    drop_events = df[n_dropped_est >= 1]
    missing = []
    for i, k in zip(
        drop_events.index.to_list(),
        n_dropped_est.loc[drop_events.index].astype(int).to_list()
    ):
        missing.extend(range(i + 1, i + 1 + k))

    return missing


def insert_dropped_rows(df: pd.DataFrame, fps: float, missing_idx: list[int], ts_col="Timestamp"):
    df = df.copy()
    df[ts_col] = pd.to_datetime(df[ts_col], utc=True)
    df = df.sort_values(ts_col).reset_index(drop=True)
    df["recorded_idx"] = np.arange(len(df), dtype=int)

    expected_dt = pd.to_timedelta(1.0 / fps, unit="s")
    missing_set = set(missing_idx)

    rows = []
    for i in range(len(df)):
        rows.append({
            "global_idx": len(rows),
            "recorded_idx": i,
            ts_col: df.loc[i, ts_col],
            "is_dropped": False,
        })

        if (i + 1) in missing_set:
            k = 0
            j = i + 1
            while j in missing_set:
                k += 1
                j += 1

            last_ts = df.loc[i, ts_col]
            for kk in range(k):
                rows.append({
                    "global_idx": len(rows),
                    "recorded_idx": np.nan,
                    ts_col: last_ts + (kk + 1) * expected_dt,
                    "is_dropped": True,
                })

    filled = pd.DataFrame(rows)

    non_drop = filled["is_dropped"] == False
    other_cols = df.columns.difference([ts_col], sort=False)
    filled.loc[non_drop, other_cols] = df.loc[
        filled.loc[non_drop, "recorded_idx"].astype(int).values,
        other_cols
    ].to_numpy()

    return filled


# ------------------------------------------------------------
# Paths / parameters
# ------------------------------------------------------------

beh_path = r"C:\Users\psych-aalab\Desktop\zenon_frametest\20251028\beh-cam_frame-id_0.csv"
neu_path = r"C:\Users\psych-aalab\Desktop\zenon_frametest\20251028\miniscope_frame-id_0.csv"

fps = 30
tolerance_s = 0.002  # currently unused by detect_missing_indices
tol = pd.Timedelta(seconds=0.5 / fps)  # half-frame tolerance for asof matching


# ------------------------------------------------------------
# Load + fill NEURAL (master timeline)
# ------------------------------------------------------------

neu = pd.read_csv(neu_path)
neu["Timestamp"] = pd.to_datetime(neu["Timestamp"], utc=True)

neu_missing = detect_missing_indices(neu, fps=fps, tolerance_s=tolerance_s, ts_col="Timestamp")
neu_filled  = insert_dropped_rows(neu, fps=fps, missing_idx=neu_missing, ts_col="Timestamp")

# Rename neural columns for clarity and to keep master global index unambiguous
neu_filled = neu_filled.rename(columns={
    "Timestamp": "neu_ts",
    "is_dropped": "neu_dropped",
    "global_idx": "global_idx",   # keep as master name
})

# ------------------------------------------------------------
# Load + fill BEHAVIOR
# ------------------------------------------------------------

beh = pd.read_csv(beh_path)
beh["Timestamp"] = pd.to_datetime(beh["Timestamp"], utc=True)

beh_missing = detect_missing_indices(beh, fps=fps, tolerance_s=tolerance_s, ts_col="Timestamp")
beh_filled  = insert_dropped_rows(beh, fps=fps, missing_idx=beh_missing, ts_col="Timestamp")

# Rename behavior timestamp + drop flag; DROP its global_idx to avoid collisions
beh_filled = beh_filled.rename(columns={
    "Timestamp": "beh_ts",
    "is_dropped": "beh_dropped",
}).drop(columns=["global_idx"], errors="ignore")

# ------------------------------------------------------------
# Merge BEHAVIOR onto padded NEURAL timeline by timestamp
# ------------------------------------------------------------

neu_filled = neu_filled.sort_values("neu_ts").reset_index(drop=True)
beh_filled = beh_filled.sort_values("beh_ts").reset_index(drop=True)

aligned = pd.merge_asof(
    neu_filled,
    beh_filled,
    left_on="neu_ts",
    right_on="beh_ts",
    direction="nearest",   # or "backward"
    tolerance=tol,
)

# ------------------------------------------------------------
# Minimal output e
# ------------------------------------------------------------

aligned_min = aligned[["global_idx", "neu_ts", "beh_ts", "neu_dropped", "beh_dropped"]].copy()

out_min = r"C:\Users\psych-aalab\Desktop\aligned_minimal.csv"
aligned_min.to_csv(out_min, index=False)

print(aligned_min.head(20))
print("Neural dropped frames (est):", len(neu_missing))
print("Behavior dropped frames (est):", len(beh_missing))
print("Saved minimal to:", out_min)


    global_idx                              neu_ts  \
0            0    2025-10-28 21:03:43.801088+00:00   
1            1 2025-10-28 21:03:43.828211200+00:00   
2            2 2025-10-28 21:03:43.860979200+00:00   
3            3    2025-10-28 21:03:43.894400+00:00   
4            4 2025-10-28 21:03:43.927180800+00:00   
5            5 2025-10-28 21:03:43.960332800+00:00   
6            6 2025-10-28 21:03:43.993779200+00:00   
7            7 2025-10-28 21:03:44.026483200+00:00   
8            8 2025-10-28 21:03:44.059724800+00:00   
9            9 2025-10-28 21:03:44.092838400+00:00   
10          10 2025-10-28 21:03:44.125798400+00:00   
11          11 2025-10-28 21:03:44.158310400+00:00   
12          12 2025-10-28 21:03:44.191577600+00:00   
13          13    2025-10-28 21:03:44.224960+00:00   
14          14 2025-10-28 21:03:44.258406400+00:00   
15          15 2025-10-28 21:03:44.290777600+00:00   
16          16    2025-10-28 21:03:44.323968+00:00   
17          17 2025-10-28 21

In [2]:
aligned_min

,global_idx,neu_ts,beh_ts,neu_dropped,beh_dropped
0,0,2025-10-28 21:03:43.801088+00:00,NaT,False,NaN
1,1,2025-10-28 21:03:43.828211200+00:00,NaT,False,NaN
2,2,2025-10-28 21:03:43.860979200+00:00,NaT,False,NaN
3,3,2025-10-28 21:03:43.894400+00:00,NaT,False,NaN
4,4,2025-10-28 21:03:43.927180800+00:00,NaT,False,NaN
...,...,...,...,...,...
54258,54258,2025-10-28 21:33:38.586790400+00:00,2025-10-28 21:33:38.578931200+00:00,False,False
54259,54259,2025-10-28 21:33:38.619571200+00:00,2025-10-28 21:33:38.612505600+00:00,False,False
54260,54260,2025-10-28 21:33:38.652518400+00:00,2025-10-28 21:33:38.645875200+00:00,False,False
54261,54261,2025-10-28 21:33:38.685875200+00:00,2025-10-28 21:33:38.679040+00:00,False,False
